# Lab 3 — Orchestrate Foundry Agents with Python

## Scenario

In Labs 1 and 2 you built two AI agents in the Microsoft Foundry portal:

| Agent | Model | Role |
|-------|-------|------|
| **ZavaGroceriesInventoryAgent** | gpt-5-mini | CRUD operations on live store data (daily financials + hourly sales) via MCP tools |
| **SalesInsightsAgent** | gpt-4o | Analyzes store performance data + news context, produces structured recommendations |

In this lab you will:
1. **Connect** to your Foundry project from Python
2. **Test each agent individually** by sending prompts and viewing responses
3. **Chain both agents in a workflow** — SalesInsightsAgent produces recommendations, then ZavaGroceriesInventoryAgent acts on them

> **Key concept:** Agent behavior (instructions, tools, model) was defined when you created the agents in Foundry. From Python we simply *call* them — we don't need to re-define tools or instructions.

## Prerequisites

Before starting, make sure you have:

| What | Where to find it |
|------|------------------|
| **Foundry Project Endpoint** | Azure AI Foundry portal → your project → Overview → "Project endpoint" |
| **ZavaGroceriesInventoryAgent** name | Lab 1 — the agent name you typed when creating it |
| **SalesInsightsAgent** name | Lab 2 — the agent name you typed when creating it |
| Azure CLI authenticated | Run `az login` in your terminal if you haven't already |
| Python 3.10+ | Pre-installed in Codespaces |

## Step 1 — Install required packages

These are the official Microsoft SDKs for working with Foundry projects and agents.

In [1]:
%pip install azure-ai-projects azure-identity openai --quiet

Note: you may need to restart the kernel to use updated packages.


## Step 2 — Configure your project settings

Paste your Foundry project endpoint and agent names in the cell below. Replace each placeholder with your actual values.

> **Tip:** The agent names must match *exactly* what you typed in Foundry (including capitalization).

In [ ]:
# ── Paste your values here ──
PROJECT_ENDPOINT = "<your-project-endpoint>"
INVENTORY_AGENT_NAME = "ZavaGroceriesInventoryAgent"
INSIGHTS_AGENT_NAME = "SalesInsightsAgent"

# Quick validation
assert not PROJECT_ENDPOINT.startswith("<"), "❌ Replace the PROJECT_ENDPOINT placeholder with your actual endpoint"
print("✅ Settings configured")
print(f"   Endpoint:        {PROJECT_ENDPOINT}")
print(f"   Inventory agent: {INVENTORY_AGENT_NAME}")
print(f"   Insights agent:  {INSIGHTS_AGENT_NAME}")

✅ Settings configured
   Endpoint:        https://prj-lp-agents1-resource.services.ai.azure.com/api/projects/prj-lp-agents1
   Inventory agent: ZavaGroceriesInventoryAgent
   Insights agent:  SalesInsightsAgent


## Step 3 — Authenticate and connect to your Foundry project

This cell authenticates to Azure using your CLI credentials and creates two clients:
- **`project_client`** — manages project-level resources (agents, threads, etc.)
- **`openai_client`** — an OpenAI-compatible client that can call your Foundry agents

In [3]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
openai_client = project_client.get_openai_client()

# Store names in shorter variables for convenience
inventory_agent_name = INVENTORY_AGENT_NAME
insights_agent_name = INSIGHTS_AGENT_NAME

print("✅ Connected to Foundry project")

✅ Connected to Foundry project


## Step 4 — Look up your agents

Let's verify that both agents exist in your Foundry project by listing all agents and finding ours by name.

In [6]:
agents_client = project_client.agents

all_agents = list(agents_client.list())
print(f"Found {len(all_agents)} agent(s) in your project:\n")
for a in all_agents:
    print(f"  \u2022 {a.name}  (model: {a})")

# Verify our two agents exist
inventory_agent = next((a for a in all_agents if a.name == inventory_agent_name), None)
insights_agent = next((a for a in all_agents if a.name == insights_agent_name), None)

assert inventory_agent, f"\u274c Agent '{inventory_agent_name}' not found \u2014 check spelling in your .env file"
assert insights_agent, f"\u274c Agent '{insights_agent_name}' not found \u2014 check spelling in your .env file"

print(f"\n\u2705 Both agents verified!")

Found 4 agent(s) in your project:

  • SalesInsightsAgent  (model: {'object': 'agent', 'id': 'SalesInsightsAgent', 'name': 'SalesInsightsAgent', 'versions': {'latest': {'metadata': {'logo': 'Avatar_Default.svg', 'microsoft.voice-live.enabled': 'false', 'microsoft.voice-live.configuration': '{"session":{"voice":{"name":"en-US-Ava:DragonHDLatestNeural","type":"azure-standard","temperature":0.8,"rate":"1","isHdVoice":true},"inputAudioTranscription":{"model":"azure-speech","language":"auto-detect"},"turnDetection":{"type":"azure_semantic_vad","endOfUtteranceDetection":null,"removeFillerWords":true},"inputAudioNoiseReduction":null,"inputAudioEchoCancellation":null,"fillerResponse":null,"avatar":null,"proactiveEngagement":false}}', 'description': '', 'modified_at': '1773947788'}, 'object': 'agent.version', 'id': 'SalesInsightsAgent:3', 'name': 'SalesInsightsAgent', 'version': '3', 'description': '', 'created_at': 1773947808, 'definition': {'kind': 'prompt', 'model': 'gpt-4o', 'instructions':

## Step 5 — Create a helper function to call agents

We'll use the **OpenAI Responses API** to talk to our Foundry agents. This is the simplest approach — it handles tool execution (including MCP server calls) automatically on the server side.

The helper below wraps the API call so we can reuse it throughout the notebook.

In [9]:
def run_agent(agent_name: str, prompt: str) -> str:
    """Send a prompt to a Foundry agent and return the response text."""
    response = openai_client.responses.create(
        input=prompt,
        extra_body={"agent_reference": {"type": "agent_reference", "name": agent_name}},
    )
    return response.output_text

print("\u2705 run_agent() helper ready")

✅ run_agent() helper ready


## Step 6 — Test the Inventory Agent

Let's send a prompt to **ZavaGroceriesInventoryAgent** and see it use its MCP tools to query live store data.

This agent has access to 10 CRUD tools (list, get, create, update, delete) for both daily financial records and hourly sales data.

In [10]:
prompt = "List all daily financial records for Store-001. Summarize total chickens bought vs sold."

print(f"\U0001f4e4 Prompt: {prompt}\n")
print("\u23f3 Waiting for response (the agent is calling MCP tools behind the scenes)...\n")

result = run_agent(inventory_agent_name, prompt)

print(f"\U0001f4e5 Response from {inventory_agent_name}:\n")
print(result)

📤 Prompt: List all daily financial records for Store-001. Summarize total chickens bought vs sold.

⏳ Waiting for response (the agent is calling MCP tools behind the scenes)...

📥 Response from ZavaGroceriesInventoryAgent:

Here are all daily financial records for Store-001:

- 2026-02-20 (ID: 695ce6fd-42d8-473c-bfb2-32eda7c84326)
  - Chickens bought: 138
  - Purchase cost: $276.00
  - Chickens sold: 96
  - Sales revenue: $480.00
  - Gross profit: $204.00

- 2026-02-21 (ID: 7a1089b0-154d-4f2f-8511-9317aaf49f47)
  - Chickens bought: 152
  - Purchase cost: $304.00
  - Chickens sold: 115
  - Sales revenue: $555.00
  - Gross profit: $251.00

- 2026-02-22 (ID: e0076efd-2028-418a-9c3d-b814ffb3ced5)
  - Chickens bought: 119
  - Purchase cost: $238.00
  - Chickens sold: 85
  - Sales revenue: $425.00
  - Gross profit: $187.00

- 2026-02-23 (ID: 1667529c-d20d-46e8-9ece-90cec06a830f)
  - Chickens bought: 143
  - Purchase cost: $286.00
  - Chickens sold: 106
  - Sales revenue: $530.00
  - Gross pr

Let's try a second prompt to see the hourly sales data:

In [11]:
prompt = "Show me the hourly sales breakdown for Store-001 for the most recent date. Which hours had the best sell-through?"

print(f"\U0001f4e4 Prompt: {prompt}\n")
print("\u23f3 Waiting for response...\n")

result = run_agent(inventory_agent_name, prompt)

print(f"\U0001f4e5 Response from {inventory_agent_name}:\n")
print(result)

📤 Prompt: Show me the hourly sales breakdown for Store-001 for the most recent date. Which hours had the best sell-through?

⏳ Waiting for response...

📥 Response from ZavaGroceriesInventoryAgent:

Here’s the hourly sales breakdown for Store-001 on the most recent date in the system: 2026-03-01.

Hourly breakdown (hour — cooked — sold — sell‑through)
- 08:00 — 8 cooked — 4 sold — 50.00%
- 09:00 — 7 cooked — 5 sold — 71.43%
- 10:00 — 9 cooked — 9 sold — 100.00%
- 11:00 — 19 cooked — 15 sold — 78.95%
- 12:00 — 12 cooked — 10 sold — 83.33%
- 13:00 — 13 cooked — 12 sold — 92.31%
- 14:00 — 10 cooked — 9 sold — 90.00%
- 15:00 — 9 cooked — 8 sold — 88.89%
- 16:00 — 4 cooked — 4 sold — 100.00%
- 17:00 — 15 cooked — 15 sold — 100.00%
- 18:00 — 11 cooked — 11 sold — 100.00%
- 19:00 — 13 cooked — 10 sold — 76.92%
- 20:00 — 4 cooked — 3 sold — 75.00%

Best sell‑through hours
- 100% sell‑through at 10:00, 16:00, 17:00 and 18:00 (all cooked units were sold).


## Step 7 — Test the Sales Insights Agent

Now let's test **SalesInsightsAgent**. This agent uses both MCP tools (for live data) and Azure AI Search (for news context) to produce a structured insights report with actionable recommendations.

In [12]:
prompt = (
    "Analyze the complete sales performance for Store-001. "
    "Include ordering efficiency, cooking window analysis, "
    "relevant news context, and actionable recommendations."
)

print(f"\U0001f4e4 Prompt: {prompt}\n")
print("\u23f3 Waiting for response (this may take a moment \u2014 the agent is querying data + news)...\n")

result = run_agent(insights_agent_name, prompt)

print(f"\U0001f4e5 Response from {insights_agent_name}:\n")
print(result)

📤 Prompt: Analyze the complete sales performance for Store-001. Include ordering efficiency, cooking window analysis, relevant news context, and actionable recommendations.

⏳ Waiting for response (this may take a moment — the agent is querying data + news)...

📥 Response from SalesInsightsAgent:

---

## 📊 Insights Report — 2026-02-20 to 2026-03-01 — Store-001

### Summary
Store-001 performed moderately well but displayed persistent inefficiencies in sell-through ratios. External factors like bird flu, weather, and local events influenced demand patterns heavily, requiring adjustments in ordering and cooking strategies.

---

### Ordering Efficiency
| Date       | Bought | Sold | Sell-Through % | Status          |
|------------|--------|------|----------------|-----------------|
| 2026-02-20 | 138    | 96   | 69.57%         | Below target    |
| 2026-02-21 | 152    | 115  | 75.66%         | Acceptable      |
| 2026-02-22 | 119    | 85   | 71.43%         | Below target    |
| 2026-02-2

## Step 8 — Two-Agent Workflow: Insights → Inventory

Now for the key pattern: **agent chaining**. We'll run a two-step workflow:

```
+-------------------------+              +------------------------------+
| SalesInsightsAgent      |              | ZavaGroceriesInventoryAgent  |
|                         |  --output--> |                              |
| Analyze data &          |              | Review recommendations &     |
| produce action items    |              | check/apply to inventory     |
+-------------------------+              +------------------------------+
```

**How it works:**
1. We ask SalesInsightsAgent to generate specific recommendations
2. We capture its response text
3. We build a new prompt that includes those recommendations
4. We send that prompt to ZavaGroceriesInventoryAgent

This is the fundamental pattern behind all agent orchestration — one agent's output becomes another agent's input.

In [13]:
# \u2500\u2500 Step A: Ask SalesInsightsAgent for recommendations \u2500\u2500
insights_prompt = (
    "Generate a full insights report for Store-001 with specific, numbered action items "
    "for improving ordering efficiency and cooking schedules. "
    "Focus on concrete changes with specific numbers."
)

print("=" * 70)
print("STEP A \u2014 Asking SalesInsightsAgent for recommendations")
print("=" * 70)
print(f"\n\U0001f4e4 Prompt:\n{insights_prompt}\n")
print("\u23f3 Generating insights...\n")

insights_output = run_agent(insights_agent_name, insights_prompt)

print(f"\U0001f4e5 SalesInsightsAgent response:\n")
print(insights_output)

STEP A — Asking SalesInsightsAgent for recommendations

📤 Prompt:
Generate a full insights report for Store-001 with specific, numbered action items for improving ordering efficiency and cooking schedules. Focus on concrete changes with specific numbers.

⏳ Generating insights...

📥 SalesInsightsAgent response:

---

## 📊 Insights Report — 2026-02-20 to 2026-03-01 — Store-001

### Summary
Store-001 shows persistent inefficiencies in sell-through rates, averaging below optimal levels. Midday cooking windows are more efficient, while mornings and evenings generate waste. Adjusting order quantities and cooking schedules based on demand patterns can improve profitability.

### Ordering Efficiency
| Date       | Bought | Sold | Sell-Through % | Status               |
|------------|--------|------|----------------|----------------------|
| 2026-02-20 | 138    | 96   | 69.6%          | Suboptimal (<70%)    |
| 2026-02-21 | 152    | 115  | 75.7%          | Efficient            |
| 2026-02-22 |

In [14]:
# \u2500\u2500 Step B: Feed recommendations to ZavaGroceriesInventoryAgent \u2500\u2500
inventory_prompt = (
    "The following recommendations were generated by our sales analytics system. "
    "Please review each action item and use your tools to check the current "
    "inventory data for Store-001, then confirm which changes can be applied:\n\n"
    f"{insights_output}"
)

print("\n" + "=" * 70)
print("STEP B \u2014 Passing recommendations to ZavaGroceriesInventoryAgent")
print("=" * 70)
print(f"\n\U0001f4e4 Prompt (first 300 chars):\n{inventory_prompt[:300]}...\n")
print("\u23f3 Inventory agent is reviewing recommendations...\n")

inventory_output = run_agent(inventory_agent_name, inventory_prompt)

print(f"\U0001f4e5 ZavaGroceriesInventoryAgent response:\n")
print(inventory_output)


STEP B — Passing recommendations to ZavaGroceriesInventoryAgent

📤 Prompt (first 300 chars):
The following recommendations were generated by our sales analytics system. Please review each action item and use your tools to check the current inventory data for Store-001, then confirm which changes can be applied:

---

## 📊 Insights Report — 2026-02-20 to 2026-03-01 — Store-001

### Summary
S...

⏳ Inventory agent is reviewing recommendations...

📥 ZavaGroceriesInventoryAgent response:

I reviewed Store-001 data for 2026-02-20 → 2026-03-01 using the system records (daily financials + hourly sales). Summary and recommended next steps below.

What I retrieved
- Daily financial records (Store-001) for each date in the range — I confirmed chickens_bought / chickens_sold for every date. Example records:
  - 2026-02-20 — bought 138, sold 96 (record id: 695ce6fd-42d8-473c-bfb2-32eda7c84326)
    - purchase_cost: $276.00, revenue: $480.00, gross_profit: $204.00
  - 2026-02-24 — bought 152, sold 

## Validation Checklist

- [x] Connected to Foundry project from Python
- [x] Looked up both agents by name
- [x] Tested **ZavaGroceriesInventoryAgent** with live data queries
- [x] Tested **SalesInsightsAgent** with a full analysis prompt
- [x] Ran a **two-agent workflow** -- insights to inventory

## What you just built

You demonstrated the core pattern of **agent orchestration**: one agent's output becomes another agent's input. In production, this pattern scales to pipelines with many agents, conditional routing, and human-in-the-loop approvals.

## Troubleshooting

| Issue | Fix |
|-------|-----|
| `KeyError` on PROJECT_ENDPOINT | Replace the placeholder in Step 2 with your actual Foundry project endpoint |
| Agent not found assertion error | Check that agent names in Step 2 match exactly what you typed in Foundry (case-sensitive) |
| `DefaultAzureCredential` error | Run `az login` in your terminal |
| Timeout or empty response | The MCP server may be down -- verify it is running with `curl http://<IP>:8000/sse` |
| `ModuleNotFoundError` | Re-run Step 1 to install packages |